# Demo: Catalog searches with CQL2 requests

Demo of work done as part of tickets RSPY-160 and RSPY-656.   
This shows the usage of advanced temporal filters following these specifications: https://pforge-exchange2.astrium.eads.net/confluence/display/COPRS/4.+External+data+selection+policies

## 0 - Initialization

In [ ]:
import requests
import os
import pprint
import time
import pystac
from pystac import Asset, Collection, Extent, Item, SpatialExtent, TemporalExtent, ItemCollection
# Init environment before running a demo notebook.
from resources.utils import *

pp = pprint.PrettyPrinter(indent=2, width=80, sort_dicts=False, compact=True)
session = requests.Session()
auxip_client, cadip_client, catalog_client, staging_client = init_demo()

if os.getenv("RSPY_LOCAL_MODE") == "1":
    href_cadip = "http://rs-server-cadip:8000"
    href_adgs = "http://rs-server-adgs:8000"
else:
    href_cadip = href_adgs = os.environ["RSPY_WEBSITE"]
    session.cookies.set("session", os.environ["RSPY_OAUTH2_COOKIE"])

cadip_collection_id = "cadip_sentinel1"
adgs_collection_id = "adgs"
TIMEOUT = 10
collection_description = Collection(
    id=TEST_COLLECTION,
    description=None,  # rs-client will provide a default description for us
    extent=Extent(
        spatial=SpatialExtent(bboxes=[-180.0, -90.0, 180.0, 90.0]),
        temporal=TemporalExtent([start_date, stop_date]),
    ),
)

# Init the dask cluster
from resources.dask_utils import *
init_dask_cluster_staging(scale=2)

# Reload the global vars again
from resources.dask_utils import *

# Use the staging cluster
dask_gateway = dask_gateway_staging
dask_cluster = dask_cluster_staging

## 1 - Building catalog

The following code is taken from demo called "404_582_rsclient.ipynb".   
This section creates a catalog by staging data from AUXIP and CADIP stations.

In [ ]:
# Create a test collection 
collection = create_test_collection()

# Get all the items from the collection "cadip_sentinel1" found in the configuration of the CADIP station
items_collection_cadip = list(cadip_client.get_items(cadip_collection_id))
assert len(items_collection_cadip) > 0

# Request 14 items from the collection "adgs" found in the configuration of the ADGS station
items_collection_adgs = auxip_client.search(max_items = 14, collections = [adgs_collection_id])
assert len(items_collection_adgs) == 14

# Starting 2 staging processes, one from the CADIP station and one from the ADGS station
staging_resp_list = []
for items in [pystac.ItemCollection(list(items_collection_cadip)), pystac.ItemCollection(list(items_collection_adgs))]:
    staging_resp_list.append(staging_client.run_staging(items.to_dict(), TEST_COLLECTION))
    
timeout = 120

for resp in staging_resp_list:
    while timeout > 0:
        if "running" not in resp["status"]:
            break
        job_info = staging_client.get_job_info(resp["jobID"])
        # pprint.PrettyPrinter(indent=4).pprint(job_info)
        # print("\n")
        if "successful" in job_info["status"]:
            print(" ----- Job COMPLETED \n")
            break
        if "failed" in job_info["status"]:
            print("-----Job FAILED \n")
            break
        time.sleep(2)
        timeout -= 2

In [ ]:
# Check the catalog for my_test_collection
result = list(catalog_client.get_items(TEST_COLLECTION))

for item in result:
    print(f"Item {item.id} has {len(item.assets)} assets")

## 2 - Run various search requests with different filters to retrieve parts of the data

In [ ]:
valcover_filter =  {
    "op": "t_contains",
    "args": [
        {"interval": [{"property": "start_datetime"}, {"property": "end_datetime"}]},
        {"interval": ["2024-05-27T09:44:12.509000Z", "2024-05-27T09:44:13.509000Z"]}
    ]
}

# http://localhost:8003/catalog/search?collections=ecombelles_my_test_collection&filter=T_CONTAINS(INTERVAL(start_datetime,end_datetime),INTERVAL(TIMESTAMP(%272024-05-27T09:44:12.509000Z%27),TIMESTAMP(%272024-05-27T09:44:13.509000Z%27)))

params = {
    "owner_id": "ecombelles",
    "max_items": 100,
    "collections": ["my_test_collection"],
    "stac_filter": valcover_filter
}

catalog_client.search(**params)

In [ ]:
latestvalcover_filter =  {
    "op": "t_contains",
    "args": [
        {"interval": [{"property": "start_datetime"}, {"property": "end_datetime"}]},
        {"interval": ["2024-05-27T09:44:12.509000Z", "2024-05-27T09:44:13.509000Z"]}
    ]
}

# http://localhost:8003/catalog/search?collections=ecombelles_my_test_collection&filter=T_CONTAINS(INTERVAL(start_datetime,end_datetime),INTERVAL(TIMESTAMP(%272024-05-27T09:44:12.509000Z%27),TIMESTAMP(%272024-05-27T09:44:13.509000Z%27)))&sortby=-properties.created&limit=1

params = {
    "owner_id": "ecombelles",
    "collections": ["my_test_collection"],
    "stac_filter": latestvalcover_filter,
    "sortby": [
        {
            "field": "created",
            "direction": "desc"
        }
    ],
    "max_items": 1,
}

catalog_client.search(**params)

In [ ]:
valintersect_filter =  {
    "op": "t_intersects",
    "args": [
        {"interval": [{"property": "start_datetime"}, {"property": "end_datetime"}]},
        {"interval": ["2024-01-21T09:44:12.509000Z", "2024-06-26T09:44:13.509000Z"]}
    ]
}

# http://localhost:8003/catalog/search?collections=ecombelles_my_test_collection&filter=T_INTERSECTS(INTERVAL(start_datetime,end_datetime),INTERVAL(TIMESTAMP(%272024-01-21T09:44:12.509000Z%27),TIMESTAMP(%272024-06-26T09:44:13.509000Z%27)))

params = {
    "owner_id": "ecombelles",
    "max_items": 100,
    "collections": ["my_test_collection"],
    "stac_filter": valintersect_filter
}

catalog_client.search(**params)

In [ ]:
latestvalintersect_filter =  {
    "op": "t_intersects",
    "args": [
        {"interval": [{"property": "start_datetime"}, {"property": "end_datetime"}]},
        {"interval": ["2024-01-21T09:44:12.509000Z", "2024-06-26T09:44:13.509000Z"]}
    ]
}

# http://localhost:8003/catalog/search?collections=ecombelles_my_test_collection&filter=T_INTERSECTS(INTERVAL(start_datetime,end_datetime),INTERVAL(TIMESTAMP(%272024-01-21T09:44:12.509000Z%27),TIMESTAMP(%272024-06-26T09:44:13.509000Z%27)))&sortby=-properties.created&limit=1

params = {
    "owner_id": "ecombelles",
    "collections": ["my_test_collection"],
    "stac_filter": latestvalintersect_filter,
    "sortby": [{"field": "created", "direction": "desc"}],
    "max_items": 1,
}

catalog_client.search(**params)

In [ ]:
# latestvalidity

# http://localhost:8003/catalog/search?collections=ecombelles_my_test_collection&sortby=-properties.created&limit=1

params = {
    "owner_id": "ecombelles",
    "collections": ["my_test_collection"],
    "sortby": [{"field": "created", "direction": "desc"}],
    "max_items": 1,
}

catalog_client.search(**params)

## 3 - Delete the catalog collection

In [ ]:
result = catalog_client.remove_collection(TEST_COLLECTION)
assert result.json()["deleted collection"] == TEST_COLLECTION
pp.pprint(result.json())